# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. All references to entities—such as record sets, fields, and columns—are made using their unique `@id` values, in accordance with best practices for Croissant datasets.

### Dataset Source
The dataset is described by a Croissant JSON-LD file accessible at this URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

We load the dataset's Croissant schema and instantiate a `mlcroissant.Dataset` object. This will give us access to metadata and records as defined in the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's examine the available record sets, their `@id` identifiers, and which fields are present in each record set. We use `dataset.record_sets` and reference entities only by their `@id` field.

In [ ]:
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  Record Set @id: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("    Field @ids:")
        for field in fields:
            # Each field is a dict or a reference. We try to print the @id or dict.
            field_id = field.get('@id') if isinstance(field, dict) else field
            print(f"      - {field_id}")
    else:
        print("    No fields found.")

print("\nExample records from each record set:")
for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nRecord Set @id: {rs_id}")
    for i, record in enumerate(dataset.records(record_set=rs_id)):
        print(record)
        if i > 2:  # Show only up to 3 records for brevity
            break

## 3. Data Extraction

We load records from each record set into separate pandas DataFrames, using only the `@id` values for selection as per Croissant principles. All record sets discovered above are loaded and displayed.

In [ ]:
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"\nLoaded DataFrame for Record Set @id: {record_set_id}")
    print(f"Columns (@ids): {dataframes[record_set_id].columns.tolist()}")
    display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field (`@id`), filter the rows with values above a threshold, normalize the values, and if available, group by a categorical field. All fields are referred to by their `@id`.

In [ ]:
# For demonstration, let's select the first record set and try to find a suitable numeric field
import numpy as np

if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id].copy()
    print(f"Performing EDA on Record Set @id: {rs_id}")
    
    # Guess a numeric field (@id) by dtype
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
        # Try coercion for object columns
        try:
            coerced = pd.to_numeric(df[col], errors='coerce')
            if coerced.notna().sum() > 0:
                df[col] = coerced
                numeric_field_id = col
                break
        except Exception:
            continue
    if not numeric_field_id:
        print("No numeric field found.")
    else:
        print(f"Using numeric field @id: {numeric_field_id}")
        threshold = df[numeric_field_id].dropna().quantile(0.9) if not df[numeric_field_id].empty else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to find a candidate group field (categorical)
        group_field_id = None
        for col in df.columns:
            if col == numeric_field_id:
                continue
            # Prefer a column with a small number of unique values
            if df[col].nunique() > 1 and df[col].nunique() < len(df) / 2:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
            print(f"Grouped filtered data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")

## 5. Visualization

Let's visualize a numeric field's distribution and, if categorical data is available, display groupwise bar plots using only the `@id` of the fields. Adjust plot size and style for clarity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id:
    fig, ax = plt.subplots(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field_id}")
    ax.set_xlabel(numeric_field_id)
    plt.show()

    # If a group field exists, show groupwise bar plot
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we used the Croissant schema and the `mlcroissant` library to load, examine, and analyze structured survey and regression output data for rangeland management adoption in Northern Kenya. By always referencing fields and record sets by their `@id`, we ensured clarity and consistency when interacting with the data. Further detailed analysis can be conducted depending on the research and policy questions, using the structured approach outlined here.